In [1]:
# basic imports
import pandas as pd
import sys
import numpy as np
import pandas as pd
import scanpy as sc
# add `Tangram` to path
import tangram as tg

In [3]:
labels = pd.read_csv("/disk1/home/zhangdaijun/Data/mouse_brain/adult_mouse_brain_sc_STitch3D_cellocation/snRNA_annotation_astro_subtypes_refined59_20200823.csv", index_col=0)

labels = labels.reindex(index=ad_sc.obs_names)
ad_sc.obs[labels.columns] = labels
ad_sc = ad_sc[~ad_sc.obs['annotation_1'].isna(), :]
ad_sc.obs['celltype'] = ad_sc.obs['annotation_1']

In [4]:
celltype_key = 'celltype'

In [5]:
# use raw count both of scrna and spatial
sc.pp.normalize_total(ad_sc)
celltype_counts = ad_sc.obs[celltype_key].value_counts()
celltype_drop = celltype_counts.index[celltype_counts < 2]
print(f'Drop celltype {list(celltype_drop)} contain less 2 sample')
ad_sc = ad_sc[~ad_sc.obs[celltype_key].isin(celltype_drop),].copy()
sc.tl.rank_genes_groups(ad_sc, groupby=celltype_key, use_raw=False)
markers_df = pd.DataFrame(ad_sc.uns["rank_genes_groups"]["names"]).iloc[0:200, :]
print(markers_df)
genes_sc = np.unique(markers_df.melt().value.values)
print(genes_sc)
genes_st = ad_sp.var_names.values
genes = list(set(genes_sc).intersection(set(genes_st)))

Drop celltype [] contain less 2 sample
              Astro_AMY       Astro_AMY_CTX           Astro_CTX  \
0    ENSMUSG00000005089  ENSMUSG00000022112  ENSMUSG00000005089   
1    ENSMUSG00000022112  ENSMUSG00000024109  ENSMUSG00000022112   
2    ENSMUSG00000039375  ENSMUSG00000059974  ENSMUSG00000061080   
3    ENSMUSG00000024109  ENSMUSG00000005089  ENSMUSG00000059974   
4    ENSMUSG00000069769  ENSMUSG00000061080  ENSMUSG00000024109   
..                  ...                 ...                 ...   
195  ENSMUSG00000019856  ENSMUSG00000001700  ENSMUSG00000061740   
196  ENSMUSG00000021420  ENSMUSG00000021379  ENSMUSG00000116417   
197  ENSMUSG00000015766  ENSMUSG00000075012  ENSMUSG00000019856   
198  ENSMUSG00000020021  ENSMUSG00000071856  ENSMUSG00000040957   
199  ENSMUSG00000022122  ENSMUSG00000086503  ENSMUSG00000059495   

              Astro_HPC          Astro_HYPO           Astro_STR  \
0    ENSMUSG00000005089  ENSMUSG00000021010  ENSMUSG00000005089   
1    ENSMUSG0000002410

In [6]:
tg.pp_adatas(ad_sc, ad_sp, genes=genes)

INFO:root:0 training genes are saved in `uns``training_genes` of both single cell and spatial Anndatas.
INFO:root:0 overlapped genes are saved in `uns``overlap_genes` of both single cell and spatial Anndatas.
INFO:root:uniform based density prior is calculated and saved in `obs``uniform_density` of the spatial Anndata.
INFO:root:rna count based density prior is calculated and saved in `obs``rna_count_based_density` of the spatial Anndata.


In [7]:
ad_map = tg.map_cells_to_space(
                   ad_sc,
                   ad_sp,
                   mode='clusters',
                   cluster_label=celltype_key)

INFO:root:Allocate tensors for mapping.
INFO:root:Begin training with 0 genes and rna_count_based density_prior in clusters mode...
INFO:root:Printing scores every 100 epochs.


KL reg: 0.057
KL reg: 0.000
KL reg: 0.000
KL reg: 0.000
KL reg: 0.000
KL reg: 0.000
KL reg: 0.000
KL reg: 0.000
KL reg: 0.000
KL reg: 0.000


INFO:root:Saving results..


In [8]:

tg.project_cell_annotations(ad_map, ad_sp, annotation=celltype_key)

celltype_density = ad_sp.obsm['tangram_ct_pred']
celltype_density = (celltype_density.T/celltype_density.sum(axis=1)).T

# celltype_density.to_csv(output_file_path)

INFO:root:spatial prediction dataframe is saved in `obsm` `tangram_ct_pred` of the spatial AnnData.


In [10]:
np.save('/disk1/home/zhangdaijun/Result/Result_mouse_brain/map_tangram.npy', celltype_density)